<a href="https://colab.research.google.com/github/bemakerorg/AIoT_Book_II_RF/blob/main/AIoT_RF_Book_ES_23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELLA 1 - CREARE DATASET IMMAGINI DA PIXABAY
# ========================================
# CONFIGURAZIONE DATASET - MODIFICA QUI
# ========================================

# API key di Pixabay per accedere al servizio (registrati su pixabay.com)
PIXABAY_API_KEY = "LA TUA API KEY DI PIXABAY"

# Numero totale di immagini da scaricare per il dataset
NUM_IMAGES = 1200

# Larghezza delle immagini finali in pixel
IMAGE_WIDTH = 224

# Altezza delle immagini finali in pixel
IMAGE_HEIGHT = 224

# Numero di canali colore: 3 per RGB, 1 per Grayscale
NUM_CHANNELS = 3

# Nome identificativo del dataset che verrà creato
DATASET_NAME = "no_face"

# ========================================

In [ ]:
# CELLA 2 - IMPORTA LE LIBRERIE E I PACCHETTI NECESSARI
# ========================================
# Installa le librerie Python necessarie per il funzionamento dello script
# Pillow: per manipolazione e processamento delle immagini
# requests: per effettuare chiamate HTTP alle API
!pip install Pillow requests

# Importa il modulo os per operazioni sul sistema operativo (creazione cartelle, percorsi)
import os
# Importa requests per effettuare richieste HTTP alle API di Pixabay
import requests
# Importa time per gestire pause e timestamp
import time
# Importa PIL (Python Imaging Library) per manipolazione immagini
from PIL import Image
# Importa BytesIO per gestire stream di byte in memoria
from io import BytesIO
# Importa zipfile per creare archivi compressi
import zipfile
# Importa files da google.colab per scaricare file sul PC locale
from google.colab import files
# Importa random per mescolare liste e aggiungere casualità
import random

In [ ]:
# CELLA 3 - DEFINIZIONE DELLA CLASSE PIXABAY PER IL DOWNLOAD
# ========================================
class PixabayDownloader:
    """
    Classe per gestire il download di immagini da Pixabay
    Gestisce ricerca, download e processamento delle immagini
    """

    def __init__(self, api_key):
        """
        Inizializza il downloader con la chiave API

        Args:
            api_key (str): Chiave API di Pixabay per autenticazione
        """
        # Salva la chiave API fornita dall'utente
        self.api_key = api_key

        # URL base delle API di Pixabay per le ricerche
        self.base_url = "https://pixabay.com/api/"

    def search_images(self, query, per_page=200, page=1, category=""):
        """
        Cerca immagini su Pixabay usando l'API

        Args:
            query (str): Termine di ricerca per le immagini
            per_page (int): Numero di risultati per pagina (max 200)
            page (int): Numero di pagina dei risultati
            category (str): Categoria specifica di Pixabay

        Returns:
            dict: Risposta JSON dell'API con i risultati della ricerca
        """
        # Crea il dizionario dei parametri per la richiesta API
        params = {
            'key': self.api_key,              # Chiave API per autenticazione
            'q': query,                       # Query di ricerca
            'image_type': 'photo',            # Tipo: solo fotografie
            'orientation': 'all',             # Orientamento: tutti (verticale, orizzontale, quadrato)
            'category': category,             # Categoria specifica se fornita
            'safesearch': 'true',            # Filtro contenuti sicuri attivato
            'per_page': per_page,            # Numero risultati per pagina
            'page': page,                    # Pagina corrente
            'min_width': 200,                # Larghezza minima immagini
            'min_height': 200                # Altezza minima immagini
        }

        try:
            # Effettua la richiesta GET all'API di Pixabay
            response = requests.get(self.base_url, params=params, timeout=10)

            # Verifica che la richiesta sia andata a buon fine (status 200)
            response.raise_for_status()

            # Converte la risposta JSON in dizionario Python
            return response.json()

        except Exception as e:
            # In caso di errore, stampa il messaggio e ritorna None
            print(f"Errore nella ricerca: {e}")
            return None

    def download_and_resize_image(self, url, filename, width, height, channels):
        """
        Scarica un'immagine da URL e la ridimensiona secondo le specifiche

        Args:
            url (str): URL dell'immagine da scaricare
            filename (str): Percorso dove salvare l'immagine processata
            width (int): Larghezza target dell'immagine
            height (int): Altezza target dell'immagine
            channels (int): Numero di canali colore (1=grayscale, 3=RGB)

        Returns:
            bool: True se il download e processamento sono riusciti, False altrimenti
        """
        try:
            # Crea headers HTTP per simulare un browser reale
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }

            # Scarica l'immagine dall'URL con timeout di 15 secondi
            response = requests.get(url, headers=headers, timeout=15)

            # Verifica che il download sia riuscito
            response.raise_for_status()

            # Apri l'immagine dai byte scaricati usando PIL
            img = Image.open(BytesIO(response.content))

            # Gestisce la conversione dei canali colore
            if channels == 1:
                # Converti in scala di grigi (Luminance)
                img = img.convert('L')
                # Colore di sfondo bianco per grayscale
                canvas_color = 255
            else:
                # Converti in RGB (Red, Green, Blue)
                img = img.convert('RGB')
                # Colore di sfondo bianco per RGB
                canvas_color = (255, 255, 255)

            # Ridimensiona l'immagine mantenendo le proporzioni originali
            # thumbnail() ridimensiona senza distorcere l'immagine
            img.thumbnail((width, height), Image.Resampling.LANCZOS)

            # Crea un canvas (tela) delle dimensioni esatte richieste
            if channels == 1:
                # Canvas grayscale
                canvas = Image.new('L', (width, height), canvas_color)
            else:
                # Canvas RGB
                canvas = Image.new('RGB', (width, height), canvas_color)

            # Calcola la posizione per centrare l'immagine ridimensionata
            x = (width - img.width) // 2   # Posizione X centrale
            y = (height - img.height) // 2 # Posizione Y centrale

            # Incolla l'immagine ridimensionata al centro del canvas
            canvas.paste(img, (x, y))

            # Salva l'immagine finale in formato JPEG con qualità 90%
            canvas.save(filename, 'JPEG', quality=90)

            # Ritorna True per indicare successo
            return True

        except Exception as e:
            # In caso di errore, stampa il messaggio e ritorna False
            print(f"Errore download {url}: {e}")
            return False

In [ ]:
# CELLA 4 - FUNZIONE DI CREAZIONE DATASET
# ========================================
def create_dataset(api_key, num_images, width, height, channels, dataset_name):
    """
    Funzione principale per creare il dataset scaricando immagini da Pixabay

    Args:
        api_key (str): Chiave API di Pixabay
        num_images (int): Numero di immagini da scaricare
        width (int): Larghezza delle immagini
        height (int): Altezza delle immagini
        channels (int): Numero di canali colore
        dataset_name (str): Nome del dataset

    Returns:
        tuple: (directory_dataset, numero_immagini_scaricate)
    """

    # Validazione della chiave API
    if not api_key or api_key == "YOUR_PIXABAY_API_KEY":
        print("⚠️  ERRORE: API key di Pixabay non valida!")
        print("Inserisci la tua API key nella variabile PIXABAY_API_KEY")
        return None, 0

    # Validazione del numero di canali (deve essere 1 o 3)
    if channels not in [1, 3]:
        print("⚠️  ERRORE: NUM_CHANNELS deve essere 1 (grayscale) o 3 (RGB)")
        return None, 0

    # Crea un'istanza del downloader con la chiave API
    downloader = PixabayDownloader(api_key)

    # Crea il nome della directory del dataset
    dataset_dir = f"{dataset_name}_dataset"

    # Crea la directory se non esiste già
    os.makedirs(dataset_dir, exist_ok=True)

    # Lista di query di ricerca ottimizzate per immagini senza volti umani
    search_queries = [
        # Categoria: Natura e paesaggi
        "mountain landscape", "forest trees", "ocean waves", "desert sand",
        "sunset sky", "clouds nature", "lake water", "waterfall nature",
        "autumn leaves", "winter snow", "spring flowers", "summer beach",

        # Categoria: Oggetti e still life
        "vintage objects", "old books", "coffee cup", "kitchen utensils",
        "musical instruments", "art supplies", "tools workshop", "antique items",

        # Categoria: Architettura e strutture
        "architecture building", "bridge structure", "lighthouse tower",
        "church cathedral", "modern building", "historic castle", "urban street",

        # Categoria: Cibo e bevande
        "food photography", "fresh fruits", "vegetables garden", "bread bakery",
        "pasta dish", "dessert cake", "wine bottle", "tea ceremony",

        # Categoria: Arte e texture
        "abstract art", "texture pattern", "geometric shapes", "color gradient",
        "fabric texture", "wood grain", "stone surface", "metal texture",

        # Categoria: Natura morta e minimalismo
        "still life", "minimalist design", "simple objects", "clean composition"
    ]

    # Categorie specifiche di Pixabay che tipicamente non contengono volti
    categories = [
        "nature",          # Natura
        "backgrounds",     # Sfondi
        "science",         # Scienza
        "education",       # Educazione
        "food",           # Cibo
        "travel",         # Viaggi
        "buildings",      # Edifici
        "computer",       # Computer
        "industry",       # Industria
        "transportation", # Trasporti
        "music",          # Musica
        "places"          # Luoghi
    ]

    # Inizializza i contatori per le statistiche
    downloaded_count = 0  # Immagini scaricate con successo
    failed_count = 0      # Download falliti

    # Determina il tipo di canali per la visualizzazione
    channel_type = "Grayscale" if channels == 1 else "RGB"

    # Mostra la configurazione corrente del dataset
    print(f"🚀 Configurazione Dataset:")
    print(f"   📊 Immagini target: {num_images}")
    print(f"   📐 Dimensioni: {width}x{height}")
    print(f"   🎨 Tipo: {channel_type} ({channels} canali)")
    print(f"   📂 Nome dataset: {dataset_name}")
    print(f"   🔑 API Key: {'✅ Configurata' if api_key else '❌ Mancante'}")
    print()

    # Crea tutte le combinazioni possibili di query e categorie
    search_combinations = []
    for query in search_queries:
        for category in categories:
            # Aggiunge ogni coppia (query, categoria) alla lista
            search_combinations.append((query, category))

    # Mescola casualmente le combinazioni per maggiore varietà
    random.shuffle(search_combinations)

    print(f"🔍 Inizio ricerca e download da Pixabay...")

    # Itera attraverso tutte le combinazioni di ricerca
    for query, category in search_combinations:
        # Controlla se abbiamo raggiunto il numero target di immagini
        if downloaded_count >= num_images:
            break

        print(f"Cercando: '{query}' in '{category}'...")

        # Effettua la ricerca su Pixabay con la combinazione corrente
        results = downloader.search_images(query, per_page=200, category=category)

        # Controlla se la ricerca ha prodotto risultati validi
        if not results or 'hits' not in results:
            continue

        # Estrae la lista delle immagini trovate
        hits = results['hits']

        # Salta se non ci sono immagini in questa ricerca
        if len(hits) == 0:
            continue

        print(f"✅ Trovate {len(hits)} immagini")

        # Processa ogni immagine trovata nella ricerca corrente
        for hit in hits:
            # Controlla di nuovo se abbiamo raggiunto il target
            if downloaded_count >= num_images:
                break

            # Estrae l'URL dell'immagine in formato web (media qualità)
            image_url = hit.get('webformatURL', '')

            # Salta se l'URL non è disponibile
            if not image_url:
                continue

            # Crea un nome file unico con numerazione progressiva
            filename = os.path.join(dataset_dir, f"{dataset_name}_{downloaded_count+1:04d}.jpg")

            # Scarica e processa l'immagine con i parametri configurati
            if downloader.download_and_resize_image(image_url, filename, width, height, channels):
                # Incrementa il contatore delle immagini scaricate
                downloaded_count += 1

                # Mostra il progresso ogni 100 immagini
                if downloaded_count % 100 == 0:
                    print(f"📥 Progresso: {downloaded_count}/{num_images} immagini")
            else:
                # Incrementa il contatore dei fallimenti
                failed_count += 1

            # Pausa breve per rispettare i rate limits dell'API
            time.sleep(0.1)

        # Pausa più lunga tra diverse query per non sovraccaricare il server
        time.sleep(0.5)

    # Mostra le statistiche finali del download
    print(f"\n✅ Download completato!")
    print(f"📊 Statistiche finali:")
    print(f"   - Immagini scaricate: {downloaded_count}")
    print(f"   - Download falliti: {failed_count}")
    print(f"   - Dimensioni: {width}x{height} {channel_type}")

    # Ritorna la directory del dataset e il numero di immagini scaricate
    return dataset_dir, downloaded_count

In [ ]:
# CELLA 5 - FUNZIONE DI CREAZIONE DEL FILE ZIP DEL DATASET
# ========================================
def create_dataset_zip(dataset_dir, downloaded_count, width, height, channels, dataset_name):
    """
    Crea un archivio ZIP del dataset con documentazione inclusa

    Args:
        dataset_dir (str): Directory contenente le immagini
        downloaded_count (int): Numero di immagini nel dataset
        width (int): Larghezza delle immagini
        height (int): Altezza delle immagini
        channels (int): Numero di canali colore
        dataset_name (str): Nome del dataset

    Returns:
        str: Nome del file ZIP creato
    """
    print("\n📦 Creazione archivio ZIP...")

    # Determina il tipo di canali per la documentazione
    channel_type = "Grayscale" if channels == 1 else "RGB"

    # Crea il contenuto del file informativo del dataset
    info_content = f"""Dataset: {dataset_name.upper()}
{'=' * 50}
Configurazione:
- Nome: {dataset_name}_dataset
- Immagini totali: {downloaded_count}
- Dimensioni: {width}x{height} pixels
- Canali: {channels} ({channel_type})
- Formato: JPEG (qualità 90%)

Fonte: Pixabay API
Licenza: Pixabay License (royalty-free, uso commerciale consentito)
Data creazione: {time.strftime('%Y-%m-%d %H:%M:%S')}

Categorie incluse:
- Natura e paesaggi
- Oggetti e still life
- Architettura e strutture
- Cibo e bevande
- Arte e texture
- Composizioni minimaliste

Utilizzo consigliato:
- Addestramento modelli di computer vision
- Classificazione immagini senza volti
- Dataset negativo per face detection
- Background images per compositing

Note tecniche:
- Tutte le immagini sono state ridimensionate automaticamente
- Mantenimento proporzioni con padding bianco se necessario
- Filtro automatico per escludere contenuti con volti umani
- Rate limiting applicato per rispettare API limits
"""

    # Scrive il file informativo su disco
    with open('dataset_info.txt', 'w', encoding='utf-8') as f:
        f.write(info_content)

    # Crea un nome descrittivo per il file ZIP
    zip_filename = f'{dataset_name}_dataset_{width}x{height}_{channel_type.lower()}_{downloaded_count}imgs.zip'

    # Crea l'archivio ZIP con compressione DEFLATE
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:

        # Aggiunge tutte le immagini JPEG alla ZIP
        for filename in os.listdir(dataset_dir):
            # Controlla che il file sia un'immagine JPEG
            if filename.endswith('.jpg'):
                # Percorso completo del file sul disco
                file_path = os.path.join(dataset_dir, filename)

                # Aggiunge il file alla ZIP con struttura di cartelle
                zipf.write(file_path, f"{dataset_name}_dataset/{filename}")

        # Aggiunge il file di documentazione alla radice della ZIP
        zipf.write('dataset_info.txt', 'dataset_info.txt')

    print(f"✅ Archivio creato: {zip_filename}")

    # Ritorna il nome del file ZIP per il download
    return zip_filename

In [ ]:
# CELLA 6 - FUNZIONE PRINCIPALE
# ========================================
def main():
    """
    Funzione principale che orchestra tutto il processo di creazione del dataset
    Utilizza le variabili di configurazione globali definite all'inizio
    """
    # Stampa l'intestazione del programma
    print("=" * 70)
    print(f"🎯 GENERATORE DATASET {DATASET_NAME.upper()}")
    print("=" * 70)

    # Determina il tipo di canali per la visualizzazione
    channel_type = "Grayscale" if NUM_CHANNELS == 1 else "RGB"

    # Mostra la configurazione corrente leggendo le variabili globali
    print(f"📋 Configurazione corrente:")
    print(f"   🔑 API Key: {'✅ Configurata' if PIXABAY_API_KEY else '❌ Mancante'}")
    print(f"   📊 Numero immagini: {NUM_IMAGES}")
    print(f"   📐 Dimensioni: {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
    print(f"   🎨 Canali: {NUM_CHANNELS} ({channel_type})")
    print(f"   📂 Nome dataset: {DATASET_NAME}")
    print()

    # Chiama la funzione per creare il dataset passando tutti i parametri
    dataset_dir, downloaded_count = create_dataset(
        PIXABAY_API_KEY,    # Chiave API
        NUM_IMAGES,         # Numero di immagini target
        IMAGE_WIDTH,        # Larghezza immagini
        IMAGE_HEIGHT,       # Altezza immagini
        NUM_CHANNELS,       # Numero di canali
        DATASET_NAME        # Nome del dataset
    )

    # Controlla se la creazione del dataset è riuscita
    if dataset_dir and downloaded_count > 0:

        # Crea l'archivio ZIP del dataset
        zip_filename = create_dataset_zip(
            dataset_dir,        # Directory delle immagini
            downloaded_count,   # Numero di immagini scaricate
            IMAGE_WIDTH,        # Larghezza immagini
            IMAGE_HEIGHT,       # Altezza immagini
            NUM_CHANNELS,       # Numero di canali
            DATASET_NAME        # Nome del dataset
        )

        print(f"\n📱 Avvio download sul PC...")

        # Scarica il file ZIP sul computer locale tramite Google Colab
        files.download(zip_filename)

        # Mostra il messaggio di completamento con le statistiche finali
        print(f"\n🎉 OPERAZIONE COMPLETATA!")
        print(f"📁 File: {zip_filename}")
        print(f"🔢 Immagini: {downloaded_count}")
        print(f"📐 Formato: {IMAGE_WIDTH}x{IMAGE_HEIGHT} {channel_type}")
        print(f"📜 Licenza: Pixabay (uso commerciale OK)")
    else:
        # Mostra messaggio di errore se qualcosa è andato storto
        print("❌ Errore nella creazione del dataset.")
        print("Verifica la configurazione e riprova.")

In [ ]:
# CELLA 7 - LANCIO DEL PROCESSO DI CREAZIONE DATASET
# ========================================
# Punto di ingresso del programma
# Controlla se lo script viene eseguito direttamente (non importato)
if __name__ == "__main__":
    # Chiama la funzione principale per avviare tutto il processo
    main()